In [8]:
import pandas as pd
import numpy as np

# ____________________________LOAD DATA ________________________________________

patients = pd.read_csv("patients.csv")
admissions = pd.read_csv("admissions.csv")
services = pd.read_csv("services.csv")
drgcodes = pd.read_csv("drgcodes.csv")
transfers = pd.read_csv("transfers.csv")

print("Patients Shape:", patients.shape)
print("Admissions Shape:", admissions.shape)
print("Services Shape:", services.shape)

print("DRG Shape:", drgcodes.shape)
print("Transfers Shape:", transfers.shape)

Patients Shape: (364627, 6)
Admissions Shape: (546028, 16)
Services Shape: (593071, 5)
DRG Shape: (761856, 7)
Transfers Shape: (2413581, 7)


In [9]:

# ______________________________REMOVE DUPLICATES ___________________________________

patients = patients.drop_duplicates()
admissions = admissions.drop_duplicates()
services = services.drop_duplicates()
drgcodes = drgcodes.drop_duplicates()
transfers = transfers.drop_duplicates()

In [10]:

# _________________________________DATETIME CONVERSION ______________________________________

admissions["admittime"] = pd.to_datetime(admissions["admittime"])
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])

# ________________________________LENGTH OF STAY ________________________________

admissions["length_of_stay_days"] = (
    admissions["dischtime"]
    - admissions["admittime"]
).dt.days


# _________________________________30-DAY READMISSION TARGET _____________________________

admissions = admissions.sort_values(
    ["subject_id", "admittime"])

admissions["next_admission"] = (
    admissions.groupby("subject_id")["admittime"]
    .shift(-1))

admissions["days_to_next"] = (
    admissions["next_admission"]
    - admissions["dischtime"]
).dt.days

admissions["readmitted_30d"] = (
    admissions["days_to_next"]
    .between(0, 30)
    .astype(int))

print("\nReadmission Distribution")
print(admissions["readmitted_30d"].value_counts())


Readmission Distribution
readmitted_30d
0    435041
1    110987
Name: count, dtype: int64


In [11]:
# ______________________________PATIENT FEATURES ____________________________


patient_features = patients[
    [
        "subject_id",
        "gender",
        "anchor_age"
    ]
].rename(
    columns={
        "anchor_age": "age"
    })



In [12]:
# ________________________________SERVICE FEATURES ______________________________________

service_features = (
    services
    .groupby("hadm_id")
    .agg(
        service_count=(
            "curr_service",
            "count"
        ),
        primary_service=(
            "curr_service",
            "first"
        )
    )
    .reset_index())

In [13]:
#______________________________ DRG FEATURES _____________________________


drgcodes["drg_severity"] = pd.to_numeric(
    drgcodes["drg_severity"],
    errors="coerce")

drgcodes["drg_mortality"] = pd.to_numeric(
    drgcodes["drg_mortality"],
    errors="coerce")

drg_features = (
    drgcodes
    .groupby("hadm_id")
    .agg(
        drg_count=(
            "drg_code",
            "count"
        ),
        avg_drg_severity=(
            "drg_severity",
            "mean"
        ),
        avg_drg_mortality=(
            "drg_mortality",
            "mean"
        )
    )
    .reset_index())


In [14]:
# ____________________________________TRANSFER FEATURES ___________________________________


transfer_features = (
    transfers
    .groupby("hadm_id")
    .agg(
        transfer_count=(
            "transfer_id",
            "count"
        )
    )
    .reset_index()
)


In [15]:
# _____________________________________MERGE DATASETS _________________________________________

df = admissions.merge(
    patient_features,
    on="subject_id",
    how="left")

df = df.merge(
    service_features,
    on="hadm_id",
    how="left")

df = df.merge(
    drg_features,
    on="hadm_id",
    how="left")

df = df.merge(
    transfer_features,
    on="hadm_id",
    how="left")



In [16]:
# _____________________________HANDLE MISSING VALUES _______________________


numeric_columns = [
    "service_count",
    "drg_count",
    "avg_drg_severity",
    "avg_drg_mortality",
    "transfer_count"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = df[col].fillna(0)

if "primary_service" in df.columns:
    df["primary_service"] = df["primary_service"].fillna("Unknown")

In [17]:
# ____________________________SAVE OUTPUT_____________________________

df.to_csv(
    "admission_cohort.csv",
    index=False)

print("\nPreprocessing Complete")
print("Output File: admission_cohort.csv")
print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))


Preprocessing Complete
Output File: admission_cohort.csv
Final Shape: (546028, 28)

Missing Values:
deathtime             534238
days_to_next          223452
next_admission        223452
edregtime             166788
edouttime             166788
discharge_location    149818
marital_status         13619
insurance               9355
language                 775
admit_provider_id          4
dtype: int64
